In [1]:
import requests
import time
import platform
import psutil
from statistics import mean, stdev

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "ReqBrain"
N_RUNS = 5
WARMUP_RUNS = 1

PROMPT = """Write five software requirements for a login system.
Each requirement should be clear, concise, and testable."""

OPTIONS = {
    "temperature": 0.0,
}

print("===== SYSTEM INFORMATION =====")
print("Platform:", platform.platform())
print("Processor:", platform.processor())
print("Machine:", platform.machine())
print("CPU cores (logical):", psutil.cpu_count(logical=True))
print("CPU cores (physical):", psutil.cpu_count(logical=False))
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"RAM: {ram_gb:.2f} GB")

print("\n===== MODEL CONFIGURATION =====")
print("Model:", MODEL_NAME)
print("Runs:", N_RUNS)
print("Prompt length:", len(PROMPT.split()), "words")

def run_once():
    payload = {
        "model": MODEL_NAME,
        "prompt": PROMPT,
        "stream": False,
        "options": OPTIONS
    }

    start = time.time()
    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    end = time.time()
    response.raise_for_status()
    data = response.json()

    eval_count = data.get("eval_count")
    eval_duration = data.get("eval_duration")  # ns
    wall_time = end - start

    tps = None
    ms_per_token = None
    if eval_count is not None and eval_duration is not None and eval_count > 0 and eval_duration > 0:
        tps = eval_count / (eval_duration / 1e9)
        ms_per_token = (eval_duration / 1e6) / eval_count

    return {
        "tokens": eval_count,
        "tps": tps,
        "ms_per_token": ms_per_token,
        "wall_time": wall_time
    }

print("\nRunning warmup...")
for _ in range(WARMUP_RUNS):
    _ = run_once()

print("\n===== RUNNING BENCHMARK =====")
results = []

for i in range(N_RUNS):
    r = run_once()
    results.append(r)

    tps_str = f"{r['tps']:.2f}" if r["tps"] is not None else "NA"
    ms_str = f"{r['ms_per_token']:.2f}" if r["ms_per_token"] is not None else "NA"

    print(
        f"Run {i+1}: tokens={r['tokens']}, "
        f"tps={tps_str}, "
        f"ms/token={ms_str}, "
        f"wall={r['wall_time']:.2f}s"
    )

valid_tps = [r["tps"] for r in results if r["tps"] is not None]
valid_ms = [r["ms_per_token"] for r in results if r["ms_per_token"] is not None]

print("\n===== SUMMARY =====")
if len(valid_tps) > 0:
    tps_avg = mean(valid_tps)
    tps_std = stdev(valid_tps) if len(valid_tps) > 1 else 0.0
    print(f"Avg tokens/sec: {tps_avg:.2f} (std {tps_std:.2f})")
else:
    print("Avg tokens/sec: NA")

if len(valid_ms) > 0:
    ms_avg = mean(valid_ms)
    ms_std = stdev(valid_ms) if len(valid_ms) > 1 else 0.0
    print(f"Avg ms/token: {ms_avg:.2f} (std {ms_std:.2f})")
else:
    print("Avg ms/token: NA")

print("\n===== PAPER-READY SENTENCE =====")
if len(valid_tps) > 0 and len(valid_ms) > 0:
    print(
        f"In our local deployment setup on an Apple Silicon device "
        f"({psutil.cpu_count(logical=False)} CPU cores, {ram_gb:.0f} GB RAM), "
        f"inference with {MODEL_NAME} achieved {tps_avg:.2f} tokens/s "
        f"(≈{ms_avg:.2f} ms/token) on average over {len(valid_tps)} runs."
    )
else:
    print("Could not compute final metrics.")

===== SYSTEM INFORMATION =====
Platform: macOS-26.3.1-arm64-arm-64bit
Processor: arm
Machine: arm64
CPU cores (logical): 14
CPU cores (physical): 14
RAM: 24.00 GB

===== MODEL CONFIGURATION =====
Model: ReqBrain
Runs: 5
Prompt length: 16 words

Running warmup...

===== RUNNING BENCHMARK =====
Run 1: tokens=118, tps=31.38, ms/token=31.87, wall=3.82s
Run 2: tokens=118, tps=31.33, ms/token=31.92, wall=3.83s
Run 3: tokens=118, tps=31.19, ms/token=32.06, wall=3.84s
Run 4: tokens=118, tps=31.21, ms/token=32.05, wall=3.84s
Run 5: tokens=118, tps=31.27, ms/token=31.98, wall=3.83s

===== SUMMARY =====
Avg tokens/sec: 31.27 (std 0.08)
Avg ms/token: 31.98 (std 0.08)

===== PAPER-READY SENTENCE =====
In our local deployment setup on an Apple Silicon device (14 CPU cores, 24 GB RAM), inference with ReqBrain achieved 31.27 tokens/s (≈31.98 ms/token) on average over 5 runs.
